# PhenoAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)


Let's first import some packages:

In [1]:
from pathlib import Path
import inspect
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.PhenoAge)

class PhenoAge(pyagingModel):
    def __init__(self):
        super().__init__()

    def preprocess(self, x):
        """Apply Levine's natural-log transform to C-reactive protein.

        The published coefficient is fit against ln(CRP in mg/dL); users supply
        the raw measurement so the same column can feed clocks that log it
        differently.

        Notes
        -----
        CRP is clamped to ``PHENOAGE_CRP_FLOOR_MG_DL`` before the log, so a zero
        — from a below-detection reading, a constant imputer, or a column the
        input never had — yields a finite age instead of ``-inf`` propagating
        into every downstream summary.
        """
        index = crp_index(self.features)
        crp = torch.clamp(x[:, index : index + 1], min=PHENOAGE_CRP_FLOOR_MG_DL)
        return torch.cat([x[:, :index], torch.log(crp), x[:, index + 1 :]], dim=1)

    def postprocess(self, x):
        """Convert the fitted log hazard to clinical Phenotypic Age in years."""
       

In [3]:
model = pya.models.PhenoAge()
model.metadata.pop("version", None)
model.version = pya.__version__

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "phenoage"
model.metadata["data_type"] = "clinical biomarkers"  # Paper: blood chemistry
model.metadata["species"] = "Homo sapiens"  # Paper: Homo sapiens
model.metadata["year"] = 2018
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Levine, M. E., et al. \"An epigenetic biomarker of aging for lifespan and healthspan.\" Aging 10.4 (2018): 573-591."
model.metadata["doi"] = "https://doi.org/10.18632/aging.101414"
model.metadata["notes"] = "Clinical Phenotypic Age combines chronological age with nine blood biomarkers. The coefficients and age conversion follow [Kuo et al. (2021), section 4.3](https://doi.org/10.1111/acel.13376). The leading 1 inside the logarithm in the 2019 erratum, used in pyaging 0.6.1, compressed scores near 141.50 years. In an unweighted check of 5,645 NHANES 2009–2010 adults, 98.2% of 0.6.1 scores fell within 0.1 years of 141.50. The Kuo equation restored the age scale while preserving mortality ranking. Supply raw C-reactive protein in mg/dL. pyaging floors it at 0.01 mg/dL before taking its natural logarithm. The Kuo calibration requires pyaging >=0.6.2."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["blood"]  # Paper: blood
model.metadata["predicts"] = ["phenotypic age"]  # Paper: phenotypic age
model.metadata["training_target"] = ["mortality"]  # Paper: all-cause mortality
model.metadata["unit"] = ["years"]  # Paper: years
model.metadata["model_type"] = "penalized hazards regression with Gompertz calibration"  # Paper: Penalized proportional hazards regression with Gompertz calibration
model.metadata["platform"] = ["clinical laboratory assays"]  # Paper: clinical laboratory assays
model.metadata["population"] = "adults"  # Paper: NHANES III adults aged 20 years or older (n=9,926)
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 10
model.metadata["citations"] = 3848
model.metadata["citations_date"] = '2026-10-02'


## Download clock dependencies

In [5]:
features = [
    "albumin",
    "creatinine",
    "glucose",
    "c_reactive_protein",
    "lymphocyte_percent",
    "mean_cell_volume",
    "red_cell_distribution_width",
    "alkaline_phosphatase",
    "white_blood_cell_count",
    "age"
]

coefs = [
    -0.0336,
    0.0095,
    0.1953,
    0.0954,
    -0.0120,
    0.0268,
    0.3306,
    0.00188,
    0.0554,
    0.0804,
]

#### Published parameters and input units

[Kuo et al. (2021), section 4.3](https://doi.org/10.1111/acel.13376) gives the log-hazard intercept as −19.907 and the alkaline phosphatase coefficient as 0.00188. The package class evaluates the published mortality-to-age equation in log space:

`141.50 + (xb + log(0.00553 * 1.51714 / 0.0076927)) / 0.09165`

The leading 1 printed in the [2019 erratum](https://doi.org/10.1371/journal.pmed.1002760) compressed pyaging 0.6.1 predictions near 141.50 years. In an unweighted check on 5,645 NHANES 2009–2010 adults, the Kuo formula had a median of 44.67 years and a 5th–95th percentile range of 15.81–81.01 years. Both formulas had an eight-year mortality AUROC of 0.8775.

Input units follow Table S1 of the [original supplement](https://cdn.aging-us.com/article/101414/supplementary/SD1/0/aging-v10i4-101414-supplementary-material-SD1.pdf). Supply raw CRP in mg/dL. The model floors and logs CRP internally.

## Load features

In [6]:
model.features = features

## Load weights into base model

In [7]:
weights = torch.tensor(coefs, dtype=torch.float64).unsqueeze(0)
intercept = torch.tensor([-19.907], dtype=torch.float64)

#### Linear model

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights
base_model.linear.bias.data = intercept

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "natural_log_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = 'mortality_to_phenoage'
model.postprocess_dependencies = None
model.metadata.update(version=model.version, preprocess=model.preprocess_name, postprocess=model.postprocess_name)


## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Levine, M. E., et al. "An epigenetic biomarker of aging for '
             'lifespan and healthspan." Aging 10.4 (2018): 573-591.',
 'citations': 3848,
 'citations_date': '2026-10-02',
 'clock_name': 'phenoage',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.18632/aging.101414',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'model_type': 'penalized hazards regression with Gompertz calibration',
 'n_features': 10,
 'notes': 'Clinical Phenotypic Age combines chronological age with nine blood '
          'biomarkers. The coefficients and age conversion follow [Kuo et al. '
          '(2021), section 4.3](https://doi.org/10.1111/acel.13376). The '
          'leading 1 inside the logarithm in the 2019 erratum, used in pyaging '
          '0.6.1, compressed scores near 141.50 years. In an un

## Normal feature ranges

In [13]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

,feature,unit,low,high
0,albumin,g/L,10.00,70.0
1,creatinine,umol/L,10.00,3000.0
2,glucose,mmol/L,1.00,60.0
3,c_reactive_protein,mg/dL,0.01,50.0
4,lymphocyte_percent,%,0.00,100.0


## Basic test

In [14]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

tensor([[498.9247],
        [498.9247],
        [498.9247],
        [498.9247],
        [498.9247],
        [498.9247],
        [498.9247],
        [498.9247],
        [498.9247],
        [498.9247]], dtype=torch.float64, grad_fn=<AddBackward0>)

In [15]:
# Independent 80-digit Decimal evaluation of Kuo et al. (2021) at fixed log hazards.
log_hazards = torch.tensor([-12.0, -9.0, -6.0], dtype=torch.float64)
published_reference = torch.tensor([11.513562158490771, 44.24678638107669, 76.98001060366262], dtype=torch.float64)
torch.testing.assert_close(model.postprocess(log_hazards), published_reference, rtol=0, atol=1e-12)


## Save torch model

In [16]:
weights_dir = Path("../weights")
weights_dir.mkdir(exist_ok=True)
torch.save(model, weights_dir / "phenoage.pt")